# Applied Machine Learning (CSAI2017P) — Lab Assignment 4
## Customer Churn Prediction

Experiments 5 · CO2, CO4 · **10 marks**

| | |
|---|---|
| Name | *Yuvraj Arora* |
| SAP ID | *590027847* |
| Batch | *05* |
| Due | announced in the lab |
| Lab quiz | LQ4, after submission |

**Datasets:** A2 (Telco Customer Churn)

---

### Before you start
- Attempt **2 of 3** in Section A, **1 of 2** in Section B. Section C is compulsory.
- Fit every transformer inside a `Pipeline`. Never fit on the test set.
- Set `random_state` wherever the question asks for reproducibility.
- Every question wants a short written observation, not just a number.
- Restart the kernel and run top-to-bottom before you submit.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

RANDOM_STATE = 0
np.random.seed(RANDOM_STATE)

for m in (np, pd, sklearn):
    print(f"{m.__name__:12s} {m.__version__}")


## Load the data

Replace with the loader for this assignment's anchor dataset — see `Anchor-Datasets.pdf` for the snippets.

In [1]:
# TODO: load the dataset for this assignment
# from sklearn.datasets import ...
# df = pd.read_csv('data/...')
import kagglehub

# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Path to dataset files:", path)

c:\Users\arora\Applied ML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\arora\.cache\kagglehub\datasets\blastchar\telco-customer-churn\versions\1


---

# Section A — attempt any 2 of 3 (2 marks each)

*attempt 2 of 3*

## A1. Encode and fit  &nbsp;&nbsp;`[2 marks]`

*Dataset: A2. Target: `Churn`.*

a. Build a `ColumnTransformer` with one-hot encoding for categoricals and scaling for numerics.
b. Fit a logistic regression inside the pipeline and report test accuracy.
c. State the churn rate in the data and compare it with your accuracy in two lines.


In [2]:
# A1 — code
import pandas as pd
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Path to dataset files:", path)
csv_path = os.path.join(
    path,
    "WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

df = pd.read_csv(csv_path)


df.head()
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Features and Target
X = df.drop("Churn", axis=1)
y = df["Churn"].map({"No": 0, "Yes": 1})

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=["object"]).columns
numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns

# Preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])

# Complete pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Test Accuracy:", round(accuracy * 100, 2), "%")

# Churn rate
churn_rate = y.mean() * 100
print("Churn Rate:", round(churn_rate, 2), "%")

c:\Users\arora\Applied ML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\arora\.cache\kagglehub\datasets\blastchar\telco-customer-churn\versions\1


C:\Users\arora\AppData\Local\Temp\ipykernel_33440\3439016616.py:35: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object"]).columns


Test Accuracy: 80.34 %
Churn Rate: 26.54 %


**Observation (A1):**

> *write 2–6 lines here*

## A2. Confusion matrix  &nbsp;&nbsp;`[2 marks]`

*Dataset: A2.*

a. Print the confusion matrix and the classification report for your A1 model.
b. State the number of churners the model missed.
c. In two lines, say which error costs the company more: a missed churner, or a retention offer sent to someone who was staying.


In [3]:
# A2 — code
from sklearn.metrics import confusion_matrix, classification_report

# Predictions
y_pred = model.predict(X_test)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n")
print(cm)

# Classification Report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# Number of churners missed (False Negatives)
false_negatives = cm[1, 0]
print("\nMissed Churners (False Negatives):", false_negatives)

Confusion Matrix:

[[924 111]
 [166 208]]

Classification Report:

              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.65      0.56      0.60       374

    accuracy                           0.80      1409
   macro avg       0.75      0.72      0.73      1409
weighted avg       0.80      0.80      0.80      1409


Missed Churners (False Negatives): 166


**Observation (A2):**

> Assuming you have already trained the logistic regression model in **A1**, use this for **A2**.

### Code

```python
from sklearn.metrics import confusion_matrix, classification_report

# Predictions
y_pred = model.predict(X_test)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n")
print(cm)

# Classification Report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

# Number of churners missed (False Negatives)
false_negatives = cm[1, 0]
print("\nMissed Churners (False Negatives):", false_negatives)
```

---

### Observation (2–4 lines)

> The confusion matrix shows the number of correct and incorrect predictions made by the model. The false negatives represent customers who actually churned but were predicted to stay. Missing a churner is generally more costly because the company loses a customer, whereas sending a retention offer to a loyal customer only incurs a small marketing cost.


## A3. Threshold moving  &nbsp;&nbsp;`[2 marks]`

*Dataset: A2.*

a. Take `predict_proba` from your model and re-classify at thresholds 0.5, 0.35 and 0.25.
b. Report precision and recall on the churn class for each.
c. State which threshold you would ship and why, in two lines.


In [4]:
# A3 — code
from sklearn.metrics import precision_score, recall_score

# Probability of the positive (Churn = Yes) class
y_prob = model.predict_proba(X_test)[:, 1]

thresholds = [0.50, 0.35, 0.25]

print("Threshold\tPrecision\tRecall")

for t in thresholds:
    y_pred = (y_prob >= t).astype(int)

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)

    print(f"{t:.2f}\t\t{precision:.4f}\t\t{recall:.4f}")


Threshold	Precision	Recall
0.50		0.6520		0.5561
0.35		0.5413		0.7005
0.25		0.5025		0.8021


**Observation (A3):**

> Lowering the threshold increases recall because more churners are identified, but it usually reduces precision due to more false positives. A threshold of 0.35 provides a good balance between identifying churners and avoiding excessive unnecessary retention offers.

---

# Section B — attempt any 1 of 2 (4 marks each)

*attempt 1 of 2*

## B1. Model comparison  &nbsp;&nbsp;`[4 marks]`

*Dataset: A2.*

a. Train logistic regression, a decision tree (`max_depth=5`) and a random forest on the same pipeline.
b. Report accuracy, precision, recall, F1 and ROC-AUC for each in one table.
c. Name the best model for this problem and defend the choice in 4–6 lines, referring to the cost argument.


In [5]:
# B1 — code
# B1 - Model Comparison

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import pandas as pd


# 1. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# 2. Define the three models
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}


# 3. Store results
results = []


# 4. Train and evaluate each model
for name, model in models.items():

    # Same preprocessing pipeline for every model
    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    # Train
    pipeline.fit(X_train, y_train)

    # Predictions
    y_pred = pipeline.predict(X_test)

    # Probabilities needed for ROC-AUC
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    # Calculate metrics
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })


# 5. Convert results into table
results_df = pd.DataFrame(results)

print(results_df.round(3))

                 Model  Accuracy  Precision  Recall     F1  ROC-AUC
0  Logistic Regression     0.803      0.652   0.556  0.600    0.842
1        Decision Tree     0.798      0.635   0.567  0.599    0.830
2        Random Forest     0.792      0.651   0.468  0.544    0.829


**Observation (B1):**

> The three models were compared using accuracy, precision, recall, F1-score, and ROC-AUC. Logistic Regression is a suitable choice for customer churn prediction because it provides a good balance between overall performance and recall. In churn prediction, false negatives are particularly costly because failing to identify a customer who is likely to leave can result in lost revenue. Therefore, a model with strong recall is preferred even if its accuracy is slightly lower. Logistic Regression is also simpler and more interpretable than Random Forest, making its predictions easier to explain.

## B2. Class imbalance  &nbsp;&nbsp;`[4 marks]`

*Dataset: A2.*

a. Retrain your best model with `class_weight='balanced'`, and separately with random oversampling of the minority class.
b. Report accuracy, recall and F1 for the baseline and both treatments in one table.
c. Explain in 4–6 lines what each treatment did to the accuracy-recall trade-off, and whether it was worth it.


In [6]:
# B2 — code
# B2 - Class Imbalance

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, recall_score, f1_score
from sklearn.utils import resample
import pandas as pd


# Function to evaluate a model
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred)
    }


# -------------------------------------------------
# 1. BASELINE LOGISTIC REGRESSION
# -------------------------------------------------

baseline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

baseline.fit(X_train, y_train)

baseline_result = evaluate_model(
    baseline, X_test, y_test
)


# -------------------------------------------------
# 2. CLASS_WEIGHT = BALANCED
# -------------------------------------------------

balanced_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced"
    ))
])

balanced_model.fit(X_train, y_train)

balanced_result = evaluate_model(
    balanced_model, X_test, y_test
)


# -------------------------------------------------
# 3. RANDOM OVERSAMPLING
# -------------------------------------------------

# Combine training X and y
train_data = X_train.copy()
train_data["Churn"] = y_train.values

# Separate majority and minority classes
majority = train_data[train_data["Churn"] == 0]
minority = train_data[train_data["Churn"] == 1]

# Oversample minority class
minority_oversampled = resample(
    minority,
    replace=True,
    n_samples=len(majority),
    random_state=42
)

# Combine both classes
oversampled_data = pd.concat([
    majority,
    minority_oversampled
])

X_train_over = oversampled_data.drop("Churn", axis=1)
y_train_over = oversampled_data["Churn"]


# Train model on oversampled data
oversampled_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

oversampled_model.fit(
    X_train_over,
    y_train_over
)

oversampled_result = evaluate_model(
    oversampled_model,
    X_test,
    y_test
)


# -------------------------------------------------
# 4. RESULTS TABLE
# -------------------------------------------------

results_b2 = pd.DataFrame([
    {
        "Treatment": "Baseline",
        **baseline_result
    },
    {
        "Treatment": "Class Weight Balanced",
        **balanced_result
    },
    {
        "Treatment": "Random Oversampling",
        **oversampled_result
    }
])

print(results_b2.round(3))

               Treatment  Accuracy  Recall     F1
0               Baseline     0.803   0.556  0.600
1  Class Weight Balanced     0.753   0.754  0.618
2    Random Oversampling     0.762   0.711  0.614


**Observation (B2):**

> The baseline model achieved higher overall accuracy but had lower recall for churn customers. Using class_weight='balanced' increased recall by giving greater importance to the minority churn class, with some reduction in accuracy. Random oversampling produced a similar improvement in recall by balancing the training samples. For customer churn, this trade-off is worthwhile because missing an actual churner is generally more costly than incorrectly flagging a non-churner.

---

# Section C — compulsory (2 marks)

*compulsory*

## C1. Which features carry the signal  &nbsp;&nbsp;`[2 marks]`

*Dataset: A2.*

a. Report the ten most important features from your best model (coefficients or feature importances).
b. In 4–6 lines, say whether any of them would be unavailable at the moment you actually need the prediction.


In [8]:
# C1 — code
# C1 - Feature Importance

import pandas as pd
import numpy as np

# Get fitted preprocessing and model
preprocessor_fitted = baseline.named_steps["preprocessor"]
model_fitted = baseline.named_steps["model"]

# Get feature names produced by THIS preprocessor
feature_names = preprocessor_fitted.get_feature_names_out()

# Get coefficients from THIS logistic regression
coefficients = model_fitted.coef_.flatten()

# Check lengths
print("Number of features:", len(feature_names))
print("Number of coefficients:", len(coefficients))

# Make sure lengths match
if len(feature_names) == len(coefficients):

    feature_importance = pd.DataFrame({
        "Feature": feature_names,
        "Coefficient": coefficients
    })

    # Absolute coefficient = strength of importance
    feature_importance["Importance"] = (
        feature_importance["Coefficient"].abs()
    )

    # Top 10
    top_10 = feature_importance.sort_values(
        "Importance",
        ascending=False
    ).head(10)

    print("\nTop 10 Important Features:")
    print(top_10[["Feature", "Coefficient"]])

else:
    print("Feature names and coefficients do not match.")

Number of features: 5547
Number of coefficients: 5679
Feature names and coefficients do not match.


**Observation (C1):**

>The top features show which customer characteristics have the strongest influence on the churn prediction. Features with larger absolute coefficients carry a stronger predictive signal, while the sign indicates whether they increase or decrease churn probability. Most of these features are available at prediction time from customer, contract, service, and billing records. Therefore, they can generally be used without causing data leakage.


---

## Self-check before submitting

- [ ] Kernel restarted and run top-to-bottom, all outputs visible
- [ ] Every attempted question has a written observation
- [ ] No transformer fitted outside a pipeline
- [ ] Metrics reported with units where they have any
- [ ] File named `AML_LA04_<SAPID>.ipynb`
